In [0]:
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"

connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [0]:
df_customer = spark.read.jdbc(jdbc_url, "dbo.DIM_CUSTOMER", properties=connection_props)
df_branch   = spark.read.jdbc(jdbc_url, "dbo.DIM_BRANCH", properties=connection_props)
df_type     = spark.read.jdbc(jdbc_url, "dbo.DIM_TRANSACTION_TYPE", properties=connection_props)
df_status   = spark.read.jdbc(jdbc_url, "dbo.DIM_TRANSACTION_STATUS", properties=connection_props)

customer_count = df_customer.count()
branch_count   = df_branch.count()
type_count     = df_type.count()
status_count   = df_status.count()


In [0]:
NUM_TRANSACTIONS = 1000

from pyspark.sql.functions import col, expr, current_timestamp, date_format, lit

df_fact_base = (
    spark.range(1, NUM_TRANSACTIONS + 1)
    .withColumn("Transaction_ID", col("id").cast("int"))
    .withColumn("Account_Number", expr("lpad(cast(cast(rand()*1e15 as bigint) as string), 15, '0')"))
    .withColumn("Customer_ID", expr(f"cast(rand()*{customer_count} as int) + 1"))
    .withColumn("Type_ID", expr(f"cast(rand()*{type_count} as int) + 1"))
    .withColumn("Amount", expr("round(rand()*4990 + 10, 2)"))
    .withColumn("Date_ID", expr("20250101 + cast(rand()*364 as int)"))
    .withColumn("Balance_After", expr("round(rand()*50000, 2)"))
    .withColumn("Branch_ID", expr(f"cast(rand()*{branch_count} as int) + 1"))
    .withColumn("Status_ID", expr(f"cast(rand()*{status_count} as int) + 1"))
    .withColumn("created_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("updated_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("is_deleted", lit(0))
    .drop("id")
)

df_fact_base.write.jdbc(
    url=jdbc_url,
    table="dbo.FACT_TRANSACTION",
    mode="overwrite",   # ✅ ONLY ONCE
    properties=connection_props
)

print("✅ Initial FACT load completed")


In [0]:
max_id_df = spark.read.jdbc(
    jdbc_url,
    "(SELECT MAX(Transaction_ID) AS max_id FROM dbo.FACT_TRANSACTION) t",
    properties=connection_props
)

START_ID = max_id_df.collect()[0]["max_id"]
print("Max Transaction_ID:", START_ID)


In [0]:
NEW_RECORDS = 499000

df_fact_incremental = (
    spark.range(1, NEW_RECORDS + 1)
    .withColumn("Transaction_ID", (col("id") + START_ID).cast("int"))
    .withColumn("Account_Number", expr("lpad(cast(cast(rand()*1e15 as bigint) as string), 15, '0')"))
    .withColumn("Customer_ID", expr(f"cast(rand()*{customer_count} as int) + 1"))
    .withColumn("Type_ID", expr(f"cast(rand()*{type_count} as int) + 1"))
    .withColumn("Amount", expr("round(rand()*4990 + 10, 2)"))
    .withColumn("Date_ID", expr("20250101 + cast(rand()*364 as int)"))
    .withColumn("Balance_After", expr("round(rand()*50000, 2)"))
    .withColumn("Branch_ID", expr(f"cast(rand()*{branch_count} as int) + 1"))
    .withColumn("Status_ID", expr(f"cast(rand()*{status_count} as int) + 1"))
    .withColumn("created_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("updated_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("is_deleted", lit(0))
    .drop("id")
)

df_fact_incremental.write.jdbc(
    url=jdbc_url,
    table="dbo.FACT_TRANSACTION",
    mode="append",     # ✅ ONLY APPEND
    properties=connection_props
)

print("✅ Incremental FACT load completed")


In [0]:
df_fact = spark.read.jdbc(jdbc_url, "dbo.FACT_TRANSACTION", properties=connection_props)
df_fact.printSchema()

In [0]:
df_fact = spark.read.jdbc(jdbc_url, "dbo.FACT_TRANSACTION", properties=connection_props)
distinct_updated_at = df_fact.select("updated_at").distinct().where("updated_at IS NOT NULL")
display(distinct_updated_at)

In [0]:
%sql
-- Get the current table location
DESCRIBE DETAIL learning_lab_catalog.silver.fact_transaction

In [0]:
%sql
-- Create backup of existing data
CREATE OR REPLACE TABLE learning_lab_catalog.silver.fact_transaction_backup AS
SELECT * FROM learning_lab_catalog.silver.fact_transaction

In [0]:
%sql
-- Drop the old table with wrong types
DROP TABLE IF EXISTS learning_lab_catalog.silver.fact_transaction

In [0]:
%sql
-- Create new table with types matching SQL Server source
CREATE TABLE learning_lab_catalog.silver.fact_transaction (
    Transaction_ID INT,
    Account_Number STRING,
    Customer_ID INT,
    Type_ID INT,
    Amount DOUBLE,
    Date_ID INT,
    Balance_After DOUBLE,
    Branch_ID INT,
    Status_ID INT,
    created_at STRING,
    updated_at STRING,
    is_deleted INT
)
USING DELTA

In [0]:
%sql
-- Insert data back with type conversions
INSERT INTO learning_lab_catalog.silver.fact_transaction
SELECT 
    Transaction_ID,
    Account_Number,
    Customer_ID,
    Type_ID,
    Amount,
    Date_ID,
    Balance_After,
    Branch_ID,
    Status_ID,
    CAST(created_at AS STRING) as created_at,
    CAST(updated_at AS STRING) as updated_at,
    CAST(is_deleted AS INT) as is_deleted
FROM learning_lab_catalog.silver.fact_transaction_backup

In [0]:
%sql
-- Verify the new schema matches source
DESCRIBE learning_lab_catalog.silver.fact_transaction

In [0]:
%sql
-- Clean up backup table
DROP TABLE IF EXISTS learning_lab_catalog.silver.fact_transaction_backup

In [0]:
%sql
-- Check total row count
SELECT COUNT(*) as total_rows
FROM learning_lab_catalog.silver.fact_transaction

In [0]:
%sql
-- View sample records
SELECT *
FROM learning_lab_catalog.silver.fact_transaction
LIMIT 10

In [0]:
%sql
select * from 

In [0]:
%sql
-- Get the most recent reconciliation ID
SELECT 
  recon_id,
  recon_table_id,
  start_ts
FROM learning_lab_catalog.lakebridge_recon.main
WHERE source_table.table_name = 'fact_transaction'
  AND target_table.table_name = 'fact_transaction'
ORDER BY start_ts DESC
LIMIT 1

In [0]:
# Get the recon_id from previous cell
recon_id_df = spark.sql("""
SELECT recon_id
FROM learning_lab_catalog.lakebridge_recon.main
WHERE source_table.table_name = 'fact_transaction'
  AND target_table.table_name = 'fact_transaction'
ORDER BY start_ts DESC
LIMIT 1
""")

if recon_id_df.count() > 0:
    latest_recon_id = recon_id_df.first()['recon_id']
    print(f"Latest recon_id: {latest_recon_id}")
    
    # Get mismatch details
    mismatch_query = f"""
    SELECT 
      d.recon_type,
      d.data
    FROM learning_lab_catalog.lakebridge_recon.details d
    JOIN learning_lab_catalog.lakebridge_recon.main m 
      ON d.recon_table_id = m.recon_table_id
    WHERE m.recon_id = '{latest_recon_id}'
      AND d.recon_type = 'mismatch'
    LIMIT 5
    """
    
    mismatch_df = spark.sql(mismatch_query)
    
    if mismatch_df.count() > 0:
        print(f"\nFound {mismatch_df.count()} mismatch records")
        print("\nAnalyzing first mismatch:")
        print("="*80)
        
        first_row = mismatch_df.first()
        data_array = first_row['data']
        
        if data_array and len(data_array) > 0:
            first_mismatch = data_array[0]
            
            # Find all columns with mismatches
            mismatched_cols = []
            for key, value in first_mismatch.items():
                if '_match' in key and value == 'false':
                    col_name = key.replace('_match', '')
                    base_val = first_mismatch.get(f"{col_name}_base", 'N/A')
                    compare_val = first_mismatch.get(f"{col_name}_compare", 'N/A')
                    mismatched_cols.append(col_name)
                    print(f"❌ {col_name}:")
                    print(f"   Source: '{base_val}'")
                    print(f"   Target: '{compare_val}'")
                    print()
            
            print("="*80)
            print(f"\nTotal mismatched columns: {len(mismatched_cols)}")
            print(f"Columns: {', '.join(mismatched_cols)}")
        else:
            print("No mismatch data found in array")
    else:
        print("No mismatch records found")
else:
    print("No reconciliation found for fact_transaction")

In [0]:
# Get metrics summary
metrics_query = f"""
SELECT 
  m.recon_id,
  met.recon_metrics.column_comparison.absolute_mismatch as mismatch_count,
  met.recon_metrics.column_comparison.mismatch_columns as mismatch_columns
FROM learning_lab_catalog.lakebridge_recon.metrics met
JOIN learning_lab_catalog.lakebridge_recon.main m 
  ON met.recon_table_id = m.recon_table_id
WHERE m.source_table.table_name = 'fact_transaction'
  AND m.target_table.table_name = 'fact_transaction'
ORDER BY m.start_ts DESC
LIMIT 1
"""

metrics_df = spark.sql(metrics_query)
if metrics_df.count() > 0:
    row = metrics_df.first()
    print(f"Recon ID: {row['recon_id']}")
    print(f"Mismatch count: {row['mismatch_count']}")
    print(f"Mismatched columns: {row['mismatch_columns']}")
    display(metrics_df)
else:
    print("No metrics found")

In [0]:
%sql
-- Clear the target table
TRUNCATE TABLE learning_lab_catalog.silver.fact_transaction

In [0]:
# Read from SQL Server
df_source = spark.read.jdbc(jdbc_url, "dbo.FACT_TRANSACTION", properties=connection_props)

print(f"Source row count: {df_source.count()}")

# Write to Databricks target
df_source.write.format("delta").mode("append").saveAsTable("learning_lab_catalog.silver.fact_transaction")

print("✅ Data reloaded from source to target")

In [0]:
%sql
SELECT 
  COUNT(*) as total_rows,
  created_at,
  updated_at,
  amount,
  balance_after
FROM learning_lab_catalog.silver.fact_transaction
GROUP BY created_at, updated_at, amount, balance_after
LIMIT 10

In [0]:
# Read sample from SQL Server source
df_source_sample = spark.read.jdbc(jdbc_url, "(SELECT TOP 5 * FROM dbo.FACT_TRANSACTION ORDER BY Transaction_ID) t", properties=connection_props)

print("Source data sample:")
print("="*80)
df_source_sample.show(5, truncate=False)

print("\nSource schema:")
df_source_sample.printSchema()

In [0]:
# Get one specific record from both source and target to compare
transaction_id = 920002  # The transaction_id from the mismatch

print(f"Comparing Transaction_ID: {transaction_id}")
print("="*80)

# Source
df_source_record = spark.read.jdbc(
    jdbc_url, 
    f"(SELECT * FROM dbo.FACT_TRANSACTION WHERE Transaction_ID = {transaction_id}) t", 
    properties=connection_props
)

if df_source_record.count() > 0:
    source_row = df_source_record.first()
    print("\nSOURCE:")
    print(f"  amount: '{source_row['Amount']}' (type: {type(source_row['Amount'])})")
    print(f"  balance_after: '{source_row['Balance_After']}' (type: {type(source_row['Balance_After'])})")
    print(f"  created_at: '{source_row['created_at']}' (type: {type(source_row['created_at'])})")
    print(f"  updated_at: '{source_row['updated_at']}' (type: {type(source_row['updated_at'])})")
else:
    print("Transaction not found in source")

# Target
df_target_record = spark.sql(f"""
    SELECT * 
    FROM learning_lab_catalog.silver.fact_transaction 
    WHERE Transaction_ID = {transaction_id}
""")

if df_target_record.count() > 0:
    target_row = df_target_record.first()
    print("\nTARGET:")
    print(f"  Amount: '{target_row['Amount']}' (type: {type(target_row['Amount'])})")
    print(f"  Balance_After: '{target_row['Balance_After']}' (type: {type(target_row['Balance_After'])})")
    print(f"  created_at: '{target_row['created_at']}' (type: {type(target_row['created_at'])})")
    print(f"  updated_at: '{target_row['updated_at']}' (type: {type(target_row['updated_at'])})")
else:
    print("Transaction not found in target")

print("\n" + "="*80)

In [0]:
%python
# Step 1: Truncate target table
spark.sql("TRUNCATE TABLE learning_lab_catalog.silver.fact_transaction")
print("✅ Target table truncated")

# Step 2: Read from SQL Server source
df_source = spark.read.jdbc(jdbc_url, "dbo.FACT_TRANSACTION", properties=connection_props)
print(f"✅ Read {df_source.count()} rows from source")

# Step 3: Write to target
df_source.write.format("delta").mode("append").saveAsTable("learning_lab_catalog.silver.fact_transaction")
print("✅ Data reloaded to target")

# Step 4: Verify
target_sample = spark.sql("SELECT updated_at, created_at FROM learning_lab_catalog.silver.fact_transaction LIMIT 5")
print("\n✅ Sample target data after reload:")
target_sample.show(truncate=False)

In [0]:
%sql
SELECT updated_at, created_at, COUNT(*) as cnt
FROM learning_lab_catalog.silver.fact_transaction
GROUP BY updated_at, created_at
LIMIT 10

In [0]:
# Delete the specific row with these exact timestamp strings
deleted_count = spark.sql("""
    DELETE FROM learning_lab_catalog.silver.fact_transaction
    WHERE updated_at = 'Jan 13 2026  9:13AM' 
      AND created_at = 'Jan 13 2026  9:05AM'
""")

print(f"✅ Deleted row(s) with updated_at='Jan 13 2026  9:13AM' and created_at='Jan 13 2026  9:05AM'")

# Verify deletion
verify_df = spark.sql("""
    SELECT COUNT(*) as remaining_count
    FROM learning_lab_catalog.silver.fact_transaction
    WHERE updated_at = 'Jan 13 2026  9:13AM' 
      AND created_at = 'Jan 13 2026  9:05AM'
""")

remaining = verify_df.first()['remaining_count']
if remaining == 0:
    print("✅ Verification: Row successfully deleted")
else:
    print(f"⚠️ Warning: {remaining} row(s) still exist with these timestamps")

In [0]:
%sql
-- Verify the row with 'Jan 13 2026  9:13AM' and 'Jan 13 2026  9:05AM' is gone
SELECT 
  updated_at, 
  created_at, 
  COUNT(*) as row_count
FROM learning_lab_catalog.silver.fact_transaction
GROUP BY updated_at, created_at
ORDER BY row_count DESC
LIMIT 20

In [0]:
# Delete records from the target table with the specified created_at and updated_at timestamps
created_at_val = "2026-01-13 09:05:00"
updated_at_val = "2026-01-13 09:13:00"

spark.sql(f"""
    DELETE FROM learning_lab_catalog.silver.fact_transaction
    WHERE created_at = '{created_at_val}' AND updated_at = '{updated_at_val}'
""")

print("✅ Record(s) deleted from target table")

# Check if records still exist
result_df = spark.sql(f"""
    SELECT * FROM learning_lab_catalog.silver.fact_transaction
    WHERE created_at = '{created_at_val}' AND updated_at = '{updated_at_val}'
""")
display(result_df)

In [0]:
%sql
select * from learning_lab_catalog.silver.fact_transaction

In [0]:
%python
from pyspark.sql.functions import expr

df = df.withColumn("updated_at", 
    expr("try_to_timestamp(updated_at, 'MMM dd yyyy  h:mma')"))

In [0]:
%sql
SELECT * FROM learning_lab_catalog.lakebridge_recon.details
     WHERE recon_id = '7fecd7ab-f325-4ac4-a936-e2bd96128023'
     ORDER BY recon_timestamp DESC

In [0]:
%sql
SELECT d.* 
FROM learning_lab_catalog.lakebridge_recon.details d
JOIN learning_lab_catalog.lakebridge_recon.main m 
  ON d.recon_table_id = m.recon_table_id
WHERE m.recon_id = '7fecd7ab-f325-4ac4-a936-e2bd96128023'
ORDER BY d.inserted_ts DESC